# Notebook 07 - Laboratorio Final

## Objetivo
Integrar en un flujo completo: lectura de CSV, limpieza, tokenizacion, padding, modelo con embeddings, entrenamiento, evaluacion y prediccion.

## Seccion A - Configuracion e hiperparametros
Puedes modificar estos valores para experimentar sin tocar el resto del pipeline.

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

config = {
    'ruta_csv': '../datasets/peliculas.csv',
    'tam_vocabulario': 1000,
    'longitud_maxima': 18,
    'dim_embedding': 24,
    'neuronas_ocultas': 16,
    'epocas': 12,
    'batch_size': 8,
    'test_size': 0.25,
    'random_state': 42,
}

print('Configuracion activa:')
for k, v in config.items():
    print(f'- {k}: {v}')

Configuracion activa:
- ruta_csv: ../datasets/peliculas.csv
- tam_vocabulario: 1000
- longitud_maxima: 18
- dim_embedding: 24
- neuronas_ocultas: 16
- epocas: 12
- batch_size: 8
- test_size: 0.25
- random_state: 42


## Seccion B - Lectura del CSV
Leemos el dataset de peliculas y revisamos estructura de columnas.

In [2]:
df = pd.read_csv(config['ruta_csv'])
print('Shape:', df.shape)
print('Columnas:', list(df.columns))
display(df.head())

Shape: (12, 3)
Columnas: ['titulo', 'resena', 'sentimiento']


,titulo,resena,sentimiento
0,Horizonte Azul,La historia fue emotiva y las actuaciones exce...,positivo
1,Noche Final,Guion confuso y ritmo lento durante casi toda ...,negativo
2,Codigo Lunar,Muy buena fotografia y final sorprendente,positivo
3,Ciudad Vacia,Dialogos forzados y personajes poco creibles,negativo
4,Ruta Infinita,Banda sonora memorable y trama bien construida,positivo


## Seccion C - Limpieza
Creamos una funcion de limpieza para la columna de resenas.

In [3]:
def limpiar_texto(texto: str) -> str:
    texto = str(texto).lower()
    texto = re.sub(r'[^a-zÃ¡Ã©Ã­Ã³ÃºÃ±\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['texto_limpio'] = df['resena'].apply(limpiar_texto)
display(df[['resena', 'texto_limpio']].head())

,resena,texto_limpio
0,La historia fue emotiva y las actuaciones exce...,la historia fue emotiva y las actuaciones exce...
1,Guion confuso y ritmo lento durante casi toda ...,guion confuso y ritmo lento durante casi toda ...
2,Muy buena fotografia y final sorprendente,muy buena fotografia y final sorprendente
3,Dialogos forzados y personajes poco creibles,dialogos forzados y personajes poco creibles
4,Banda sonora memorable y trama bien construida,banda sonora memorable y trama bien construida


## Seccion D - Tokenizacion
Convertimos texto limpio a secuencias numericas.

In [4]:
tokenizer = Tokenizer(num_words=config['tam_vocabulario'], oov_token='<OOV>')
tokenizer.fit_on_texts(df['texto_limpio'])
secuencias = tokenizer.texts_to_sequences(df['texto_limpio'])

print('Vocabulario aprendido:', len(tokenizer.word_index))
print('Ejemplo secuencia:', secuencias[0])

Vocabulario aprendido: 64
Ejemplo secuencia: [4, 9, 10, 11, 2, 12, 5, 13]


## Seccion E - Padding
Ajustamos todas las secuencias a la misma longitud.

In [5]:
X = pad_sequences(
    secuencias,
    maxlen=config['longitud_maxima'],
    padding='post',
    truncating='post'
)

y = (df['sentimiento'] == 'positivo').astype(int).values

print('Shape X:', X.shape)
print('Shape y:', y.shape)

Shape X: (12, 18)
Shape y: (12,)


## Seccion F - Split de datos
Generamos conjuntos de entrenamiento y evaluacion.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['test_size'],
    random_state=config['random_state'],
    stratify=y
)

print('Train:', X_train.shape, y_train.shape)
print('Test :', X_test.shape, y_test.shape)

Train: (9, 18) (9,)
Test : (3, 18) (3,)


## Seccion G - Modelo con Embeddings
Arquitectura base para clasificacion binaria de texto.

In [7]:
vocab_size = min(len(tokenizer.word_index) + 1, config['tam_vocabulario'])

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=config['dim_embedding'], input_length=config['longitud_maxima']),
    GlobalAveragePooling1D(),
    Dense(config['neuronas_ocultas'], activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 18, 24)            1560      
                                                                 
 global_average_pooling1d (G  (None, 24)               0         
 lobalAveragePooling1D)                                          
                                                                 
 dense (Dense)               (None, 16)                400       
                                                                 
 dense_1 (Dense)             (None, 1)                 17        
                                                                 
Total params: 1,977
Trainable params: 1,977
Non-trainable params: 0
_________________________________________________________________


## Seccion H - Entrenamiento
Entrenamos por pocas epocas para mantener el laboratorio rapido.

In [8]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=config['epocas'],
    batch_size=config['batch_size'],
    verbose=0
)

print('Accuracy final train:', round(history.history['accuracy'][-1], 4))
print('Accuracy final val  :', round(history.history['val_accuracy'][-1], 4))

Accuracy final train: 0.5556
Accuracy final val  : 0.3333


## Seccion I - Evaluacion
Calculamos metricas basicas y reporte de clasificacion.

In [9]:
prob_test = model.predict(X_test, verbose=0).ravel()
pred_test = (prob_test >= 0.5).astype(int)

acc = accuracy_score(y_test, pred_test)
print(f'Accuracy test: {acc:.4f}')
print('\nReporte de clasificacion:')
print(classification_report(y_test, pred_test, target_names=['negativo', 'positivo']))

Accuracy test: 0.3333

Reporte de clasificacion:
              precision    recall  f1-score   support

    negativo       0.00      0.00      0.00         2
    positivo       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3



c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

## Seccion J - Prediccion de nuevos textos
Aplicamos el mismo pipeline de limpieza + tokenizacion + padding antes de inferir.

In [10]:
def predecir_texto(texto: str) -> tuple[str, float]:
    limpio = limpiar_texto(texto)
    seq = tokenizer.texts_to_sequences([limpio])
    x_nuevo = pad_sequences(seq, maxlen=config['longitud_maxima'], padding='post', truncating='post')
    prob = float(model.predict(x_nuevo, verbose=0)[0][0])
    etiqueta = 'positivo' if prob >= 0.5 else 'negativo'
    return etiqueta, prob

ejemplos = [
    'La pelicula fue emocionante y muy bien actuada',
    'El guion fue aburrido y nada coherente'
]

for texto in ejemplos:
    etiqueta, prob = predecir_texto(texto)
    print(f"Texto: {texto}")
    print(f"Prediccion: {etiqueta} (prob={prob:.3f})")
    print('-' * 80)

Texto: La pelicula fue emocionante y muy bien actuada
Prediccion: positivo (prob=0.518)
--------------------------------------------------------------------------------
Texto: El guion fue aburrido y nada coherente
Prediccion: positivo (prob=0.516)
--------------------------------------------------------------------------------


## Desafios para el estudiante
Modifica y compara resultados cambiando:
1. Tamano del vocabulario (`tam_vocabulario`).
2. Longitud maxima (`longitud_maxima`).
3. Dimension del embedding (`dim_embedding`).
4. Numero de neuronas ocultas (`neuronas_ocultas`).
5. Epocas de entrenamiento (`epocas`).

Pregunta guia: Que configuracion mejora mas la generalizacion sin sobreajuste?